# Module 04 — Receptor Sequence Extraction and Embedding

**Purpose:** Extract four Xcc (*Xanthomonas campestris* pv. *campestris*) receptor proteins from the Module 02 host proteome and embed them with ESM-2. TonB, ExbB, and ExbD1 are required for phiL7 infection. ExbD2 is extracted and embedded for completeness — per Hung et al. 2003 (BBRC 302:878-884), *exbD2* mutation does NOT affect phiL7 infection; ExbD2 serves as a negative control in our receptor panel, confirming receptor specificity. (ExbD2 保留 embed，作为阴性对照。Hung 2003 确认 exbD2 突变不影响 phiL7 感染。)

**目的：** 从 Module 02 的宿主蛋白质组中提取四种 Xcc 受体蛋白（TonB、ExbB、ExbD1、ExbD2），并用 ESM-2 生成嵌入向量。其中 TonB、ExbB、ExbD1 是 phiL7 侵染所必需的；ExbD2 突变不影响 phiL7 侵染（Hung et al. 2003, BBRC 302:878-884），作为阴性对照保留，用于验证受体特异性。

**Reference / 参考文献:**
- Hung, C.-H. et al. (2003) "Involvement of the *tonB*-*exbBD1* genes in infection of *Xanthomonas campestris* phage phiL7." *Biochem Biophys Res Commun* 302:878-884. (exbD2 mutation does NOT affect infection — ExbD2 is a negative control.)
- Lin et al. (2023) *Science* 379(6637):1123-1130. DOI: [10.1126/science.ade2574](https://doi.org/10.1126/science.ade2574)
- Pfam PF03544 — TonB plug domain: https://www.ebi.ac.uk/interpro/entry/pfam/PF03544/

In [ ]:
# Cell 2: imports, versions, seeds / 导入库、版本、随机种子
import random
import sys
from pathlib import Path

import esm
import numpy as np
import pandas as pd
import torch
from Bio import SeqIO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# REPO_ROOT from notebook location (processes/) / 以 notebook 位置（processes/）定位根目录
REPO_ROOT = Path.cwd().resolve().parents[1]
MODULE_DIR = REPO_ROOT / '04_protein_embedding'
sys.path.insert(0, str(MODULE_DIR / 'processes'))

from esm2_utils import load_model, embed_sequences, save_npz, load_fasta

print('REPO_ROOT :', REPO_ROOT)
print('ESM       :', esm.__version__)
print('torch     :', torch.__version__)
print('numpy     :', np.__version__)

## Receptor Identification Strategy / 受体识别策略

We use a tiered approach to locate TonB/ExbB/ExbD1/ExbD2 in the host proteome:

采用分层策略在宿主蛋白质组中定位 TonB/ExbB/ExbD1/ExbD2：

1. **Primary:** Match known NCBI accessions from Hung et al. 2003 (WP_011035266–269).
   **首选：** 使用 Hung et al. 2003 中的已知 NCBI 登录号匹配（WP_011035266–269）。

2. **Fallback A:** Keyword regex search in FASTA headers (`tonB`, `ExbB`, `ExbD`).
   **备选 A：** 在 FASTA 序列头部用关键词正则匹配。

3. **Fallback B:** HMM scan via Pfam PF03544 (TonB plug domain) — requires `hmmer` installed.
   **备选 B：** 使用 Pfam PF03544 进行 HMM 扫描（需安装 `hmmer`）。

The pre-extracted file `inputs/xcc_receptors.faa` was produced by the primary strategy and is already committed.

预提取文件 `inputs/xcc_receptors.faa` 已用首选策略生成并提交。

In [ ]:
# Cell 3: receptor target specification
# 第 3 格：受体目标规格（来自 Hung et al. 2003）

# Known NCBI protein accessions for phiL7 receptor operon (Hung et al. 2003, BBRC 302:878-884)
# phiL7 受体操纵子已知 NCBI 蛋白质登录号（Hung et al. 2003, BBRC 302:878-884）
RECEPTOR_TARGETS = {
    'WP_011035266.1': ('GCF_000007145.1_tonB',  'energy transducer TonB'),
    'WP_011035267.1': ('GCF_000007145.1_exbB',  'TonB-system energizer ExbB'),
    'WP_011035268.1': ('GCF_000007145.1_exbD1', 'ExbD/TolR family protein (ExbD1)'),
    'WP_011035269.1': ('GCF_000007145.1_exbD2', 'ExbD/TolR family protein (ExbD2)'),
}

# Keyword fallback patterns / 关键词回退正则（区分大小写不敏感）
import re
KEYWORD_PATTERNS = [
    re.compile(r'\btonB\b', re.IGNORECASE),
    re.compile(r'\bExbB\b', re.IGNORECASE),
    re.compile(r'\bExbD\b', re.IGNORECASE),
]
print('Target receptors / 目标受体:')
for acc, (rid, desc) in RECEPTOR_TARGETS.items():
    print(f'  {acc} → {rid} ({desc})')

In [ ]:
# Cell 4: extract receptors from host proteome
# 第 4 格：从宿主蛋白质组中提取受体序列

# Primary: Module 02 annotation output / 首选：Module 02 注释输出
M02_PROTEINS = REPO_ROOT / '02_annotation' / 'outputs' / 'host_proteins' / 'Xanthomonas_campestris_33913' / 'proteins.faa'
# Fallback: raw NCBI dump / 备选：原始 NCBI 数据
RAW_PROTEINS = REPO_ROOT / '00_raw_data' / 'bacteria' / 'GCF_000007145.1' / 'proteins.faa'

host_faa = M02_PROTEINS if M02_PROTEINS.exists() else RAW_PROTEINS
print(f'Reading host proteome: {host_faa}')

found = {}
keyword_hits = {}
for rec in SeqIO.parse(str(host_faa), 'fasta'):
    # Strategy 1: exact accession match / 策略 1：精确登录号匹配
    if rec.id in RECEPTOR_TARGETS:
        found[rec.id] = rec
    # Strategy 2: keyword regex in header / 策略 2：标题关键词正则
    elif any(p.search(rec.description) for p in KEYWORD_PATTERNS):
        keyword_hits[rec.id] = rec

print(f'Found by accession / 按登录号找到: {len(found)}/{len(RECEPTOR_TARGETS)}')
print(f'Found by keyword   / 按关键词找到: {len(keyword_hits)} additional')
for acc, rec in found.items():
    rid, _ = RECEPTOR_TARGETS[acc]
    print(f'  ✓ {rid}: {len(rec.seq)} aa')

In [ ]:
# Cell 5: write extracted receptor FASTA (INTERFACE §FASTA conventions)
# 第 5 格：写出提取的受体 FASTA（符合 INTERFACE §FASTA 规范）

out_faa = MODULE_DIR / 'inputs' / 'xcc_receptors.faa'
out_faa.parent.mkdir(exist_ok=True)

with open(out_faa, 'w') as fh:
    for acc, (receptor_id, desc) in RECEPTOR_TARGETS.items():
        if acc not in found:
            print(f'WARNING: {acc} not found in proteome')
            continue
        seq = str(found[acc].seq).upper()
        # INTERFACE header format: ><id> | source=<acc> | length=<aa> | <free_text>
        fh.write(f'>{receptor_id} | source={acc} | length={len(seq)} | {desc}\n')
        for i in range(0, len(seq), 60):
            fh.write(seq[i:i+60] + '\n')

print(f'Written: {out_faa}')
for rec in SeqIO.parse(str(out_faa), 'fasta'):
    print(f'  {rec.id}: {len(rec.seq)} aa')

In [ ]:
# Cell 6: load model and embed receptors
# 第 6 格：加载模型并嵌入受体序列

MODEL_NAME = 'esm2_t6_8M_UR50D'  # CPU-safe default / CPU 安全默认
model, alphabet = load_model(MODEL_NAME)

rec_seqs = load_fasta(out_faa)
print(f'Embedding {len(rec_seqs)} receptor sequences with {MODEL_NAME} …')

rec_emb = embed_sequences(
    model, alphabet, rec_seqs,
    pooling='mean', batch_size=4,
    model_name=MODEL_NAME,
)
print('Embedding shape / 嵌入形状:', rec_emb['array'].shape)
print('Lengths         / 序列长度 :', rec_emb['lengths'])
print('seq_ids         / 序列 ID  :', list(rec_emb['seq_ids']))

In [ ]:
# Cell 7: save receptor embeddings
# 第 7 格：保存受体嵌入

out_npz = MODULE_DIR / 'outputs' / f'embeddings_{MODEL_NAME}_xcc_receptors.npz'
save_npz(out_npz, rec_emb)
print('Saved:', out_npz)

# Sanity check / 完整性检查
import numpy as np
arr = rec_emb['array']
assert arr.shape == (4, model.embed_dim), f'Shape: {arr.shape}'
assert arr.dtype == np.float32
assert not np.any(np.isnan(arr))
for i, (sid, seq) in enumerate(rec_seqs):
    assert rec_emb['lengths'][i] == len(seq)
print('Sanity checks passed / 完整性检查通过 ✓')